# Ingestion 2: APIs, pagination and retries

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/ingestion/ingestion-02-apis.ipynb)

Run the extraction against a mock transport without network calls.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q httpx

## Setup and sample data

An in-process HTTP transport returns a rate limit, then two pages. The retry waits are recorded instead of slept.

In [ ]:
import httpx

calls = []
waits = []


def api(request):
    cursor = request.url.params.get("cursor")
    calls.append(cursor)
    if len(calls) == 1:
        return httpx.Response(429, headers={"Retry-After": "2"})
    if cursor is None:
        return httpx.Response(200, json={"rows": [{"id": 1}], "next": "second"})
    return httpx.Response(200, json={"rows": [{"id": 2}], "next": None})

## Follow cursors and retry boundedly

Treat the next cursor as opaque. Retry a rate limit up to a fixed limit and respect the server-provided delay when it uses delta seconds.

**Where it matters**

Extract complete pages while honoring the API quota.

**Watch out for**

Real Retry-After can also be an HTTP date. Handle that form, cap waits, add jitter where appropriate and detect repeated cursors.

In [ ]:
def read_page(client, cursor):
    params = {"cursor": cursor} if cursor is not None else {}
    for _ in range(3):
        response = client.get("/orders", params=params)
        if response.status_code != 429:
            response.raise_for_status()
            return response.json()
        waits.append(int(response.headers.get("Retry-After", "1")))
    raise RuntimeError("Rate limit retry budget exhausted")


with httpx.Client(base_url="https://example.invalid", transport=httpx.MockTransport(api)) as client:
    cursor = None
    rows = []
    seen = set()
    while True:
        payload = read_page(client, cursor)
        rows.extend(payload["rows"])
        cursor = payload["next"]
        if cursor is None:
            break
        if cursor in seen:
            raise ValueError("API repeated a cursor")
        seen.add(cursor)
print("IDs:", [row["id"] for row in rows])
print("requests:", calls, "retry delays:", waits)

**What to notice:** The first page is requested twice, then the second page once. The result includes both IDs and one two-second wait.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Why is a retry budget needed if the endpoint always sends 429?

In [ ]:
# Try your solution here.

In [ ]:
print("It lets the pipeline fail visibly instead of retrying indefinitely.")

## Key takeaways

- Extract complete pages while honoring the API quota.

**Next:** [Ingestion 3: Object storage](ingestion-03-object-storage.ipynb)